# Diabetes RAG – Ingestion, Chunking, Embeddings & Retrieval

Loads the two diabetes guideline PDFs from GitHub, cleans and organizes them by section, creates page-aware chunks, stores embeddings in Chroma, and tests retrieval.

## 1. Download the PDFs from GitHub

In [1]:
!git clone https://github.com/Nourmohamed904/Diabetes_RAG_Hackathon.git

Cloning into 'Diabetes_RAG_Hackathon'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 61 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 855.58 KiB | 10.43 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
import os

PDF_FOLDER = "/content/Diabetes_RAG_Hackathon/data"

if not os.path.isdir(PDF_FOLDER):
    raise FileNotFoundError(
        f"PDF folder not found: {PDF_FOLDER}\n"
        "Check the repository structure or update PDF_FOLDER."
    )

PDF_PATHS = sorted(
    os.path.join(PDF_FOLDER, file)
    for file in os.listdir(PDF_FOLDER)
    if file.lower().endswith(".pdf")
)

print("Number of PDFs:", len(PDF_PATHS))
for pdf in PDF_PATHS:
    print("-", os.path.basename(pdf))

Number of PDFs: 2
- Type-1 diabetes.pdf
- Type-2 diabetes.pdf


## 2. Install and import the required libraries

In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb fastembed pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/6

In [4]:
import os
import re
import shutil
from collections import Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/tmp/ipykernel_568/1583142254.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Load the PDFs and add metadata

In [5]:
all_pages = []

for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    document_name = os.path.basename(pdf_path)

    for page in pages:
        page.metadata["document_name"] = document_name
        page.metadata["page_number"] = page.metadata.get("page", 0) + 1

    all_pages.extend(pages)

print("Total PDFs:", len(PDF_PATHS))
print("Total pages:", len(all_pages))

Total PDFs: 2
Total pages: 195


## 4. Clean the extracted text

In [6]:
def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for page in all_pages:
    page.page_content = clean_text(page.page_content)

print("Cleaning completed.")

Cleaning completed.


### 4b. Remove repeated boilerplate (copyright footer / page-of-64 lines)

Every page repeats the same NICE copyright notice and a "Page X of NN" footer, and often the guideline title as a running header. This text carries no medical meaning, but it gets embedded into every chunk that touches a page boundary, diluting the chunk's embedding and pushing genuinely relevant chunks further down the similarity ranking. Stripping it before chunking keeps each chunk focused on actual guideline content.

In [7]:
# ==========================================
# 5. Remove repeated boilerplate
# ==========================================

BOILERPLATE_PATTERNS = [
    # NICE copyright/footer
    r"©\s*NICE\s*\d{4}\.\s*All rights reserved\.\s*"
    r"(?:https?://\S+\s*)?"
    r"(?:See\s+notice-of-rights\s*\)?\.?)?",

    # Page X of Y
    r"Page\s+\d+\s+of\s+\d+",

    # Running headers
    r"Type\s+1\s+diabetes\s+in\s+adults:\s*"
    r"diagnosis\s+and\s+management\s*\(NG17\)",

    r"Type\s+2\s+diabetes\s+in\s+adults:\s*"
    r"management\s*\(NG28\)",
]


# Compile each pattern separately.
# This is safer than using one large DOTALL regex.
_BOILERPLATE_REGEXES = [
    re.compile(pattern, flags=re.IGNORECASE)
    for pattern in BOILERPLATE_PATTERNS
]


def remove_boilerplate(text):
    """
    Remove repeated PDF headers/footers without accidentally
    deleting large parts of the medical content.
    """

    for pattern in _BOILERPLATE_REGEXES:
        text = pattern.sub(" ", text)

    # Clean spaces left after removal
    text = re.sub(r"[ \t]+", " ", text)

    # Clean excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------
# Test before / after
# ------------------------------------------

sample_index = min(16, len(all_pages) - 1)

sample_before = all_pages[sample_index].page_content

sample_after = remove_boilerplate(sample_before)

print("===== BEFORE boilerplate removal =====")
print(sample_before[-500:])

print("\n===== AFTER boilerplate removal =====")
print(sample_after[-500:])


# ------------------------------------------
# Apply to all pages
# ------------------------------------------

for page in all_pages:
    page.page_content = remove_boilerplate(page.page_content)

print(
    "\nBoilerplate removal applied to all",
    len(all_pages),
    "pages"
)

===== BEFORE boilerplate removal =====
able at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 
abnormal haemoglobin type, estimate trends in blood glucose control using 1 of 
Type 1 diabetes in adults: diagnosis and management (NG17)
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).
Page 17 of
64

===== AFTER boilerplate removal =====
with type 1 diabetes their HbA1c results after each measurement and 
have their most recent result available at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 
abnormal haemoglobin type, estimate trends in blood glucose control using 1 of 
 
 Subject to Not

In [8]:
print("Document:", all_pages[0].metadata["document_name"])
print("Page:", all_pages[0].metadata["page_number"])
print("\nSample text:\n")
print(all_pages[0].page_content[:1500])

Document: Type-1 diabetes.pdf
Page: 1

Sample text:

Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17 
 Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


## 5. Detect and clean section headings

In [9]:
def is_section_heading(line):
    line = line.strip()
    if not line:
        return False

    # Main sections such as 1.1, 1.2, 1.10
    pattern = r"^\d+\.\d+\s+.+"
    return bool(re.match(pattern, line))


def clean_section_heading(line):
    line = line.strip()

    # Example:
    # 1.1 Diagnosis and early care plan ............ 6
    # -> 1.1 Diagnosis and early care plan
    line = re.sub(r"\s*\.{3,}\s*\d+\s*$", "", line)

    return line.strip()

In [10]:
test_lines = [
    "1.1 Diagnosis and early care plan",
    "1.2 Support and individualised care",
    "1.10 Ketone monitoring and managing diabetic ketoacidosis",
    "1.1.1 Make an initial diagnosis of type 1 diabetes",
    "People with diabetes should..."
]

for line in test_lines:
    print(is_section_heading(line), "→", line)

True → 1.1 Diagnosis and early care plan
True → 1.2 Support and individualised care
True → 1.10 Ketone monitoring and managing diabetic ketoacidosis
False → 1.1.1 Make an initial diagnosis of type 1 diabetes
False → People with diabetes should...


## 6. Group pages by section

In [11]:
def get_first_content_page(pages):
    """
    Detect the first page containing a real section heading.
    """

    for page in pages:
        for line in page.page_content.splitlines():
            line = line.strip()

            if is_section_heading(line):
                return page.metadata["page_number"]

    return 1


def group_pages_by_section(pages):

    grouped_sections = []

    # Group pages by PDF/document
    pages_by_document = {}

    for page in pages:
        document_name = page.metadata["document_name"]

        pages_by_document.setdefault(
            document_name, []
        ).append(page)

    # Process each PDF independently
    for document_name, document_pages in pages_by_document.items():

        current_section = None
        current_pages = []

        # Detect first content page automatically
        first_content_page = get_first_content_page(
            document_pages
        )

        print(
            f"{document_name} -> "
            f"first content page: {first_content_page}"
        )

        for page in document_pages:

            page_number = page.metadata["page_number"]

            if page_number < first_content_page:
                continue

            for raw_line in page.page_content.splitlines():

                line = raw_line.strip()

                if not line:
                    continue

                # New section
                if is_section_heading(line):

                    if current_section is not None:
                        grouped_sections.append({
                            "document_name": document_name,
                            "section": current_section,
                            "pages": current_pages.copy()
                        })

                    current_section = clean_section_heading(line)

                    current_pages = [{
                        "page_number": page_number,
                        "text": ""
                    }]

                # Normal content
                elif current_section is not None:

                    if (
                        not current_pages
                        or current_pages[-1]["page_number"] != page_number
                    ):
                        current_pages.append({
                            "page_number": page_number,
                            "text": line
                        })

                    else:
                        if current_pages[-1]["text"]:
                            current_pages[-1]["text"] += "\n" + line
                        else:
                            current_pages[-1]["text"] = line

        # Save final section
        if current_section is not None:
            grouped_sections.append({
                "document_name": document_name,
                "section": current_section,
                "pages": current_pages.copy()
            })

    return grouped_sections


# Run
grouped_sections = group_pages_by_section(all_pages)

print("Number of grouped sections:", len(grouped_sections))

if not grouped_sections:
    raise ValueError(
        "No sections were detected. "
        "Check section-heading detection and PDF extraction."
    )

first_section = grouped_sections[0]

print("\n===== FIRST SECTION =====")
print("Document:", first_section["document_name"])
print("Section:", first_section["section"])
print("Number of pages:", len(first_section["pages"]))

print(
    "Pages:",
    [p["page_number"] for p in first_section["pages"]]
)

print("\nFirst page text:")
print(first_section["pages"][0]["text"][:1000])

Type-1 diabetes.pdf -> first content page: 3
Type-2 diabetes.pdf -> first content page: 3
Number of grouped sections: 118

===== FIRST SECTION =====
Document: Type-1 diabetes.pdf
Section: 1.1 Diagnosis and early care plan
Number of pages: 1
Pages: [3]

First page text:



**Chunk size note:** originally 850/150. Lowered to **300/60** after diagnosing Q5 ("insulin plan" question): at 850 chars, recommendation 1.7.1 (insulin regimen) was sharing a chunk with the unrelated "Levemir discontinuation" notice right after it, diluting the chunk's embedding enough that it dropped out of the top-6 results. At 300 chars, 1.7.1 gets an isolated chunk. Verified this doesn't break the other working recommendations (1.6.6 HbA1c target, 1.13.1 metformin first-line) — both still land cleanly in their own chunks at this size.

## 7. Create page-aware chunks

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def create_all_chunks(grouped_sections, splitter):
    all_chunks = []
    chunk_counter = 1

    for section in grouped_sections:
        section_text = ""
        page_boundaries = []

        for page in section["pages"]:
            start_position = len(section_text)
            section_text += page["text"] + "\n"
            end_position = len(section_text)

            page_boundaries.append({
                "page_number": page["page_number"],
                "start": start_position,
                "end": end_position,
            })

        if not page_boundaries:
            continue

        section_chunks = splitter.split_text(section_text)
        search_start = 0

        for chunk_text in section_chunks:
            chunk_start = section_text.find(chunk_text, search_start)
            if chunk_start == -1:
                chunk_start = search_start

            chunk_end = chunk_start + len(chunk_text)

            chunk_pages = [
                boundary["page_number"]
                for boundary in page_boundaries
                if boundary["end"] > chunk_start
                and boundary["start"] < chunk_end
            ]

            if not chunk_pages:
                chunk_pages = [page_boundaries[0]["page_number"]]

            chunk = Document(
                page_content=chunk_text,
                metadata={
                    "document_name": section["document_name"],
                    "section": section["section"],
                    "page_number": chunk_pages[0],
                    "page_numbers": chunk_pages,
                    "chunk_id": f"chunk_{chunk_counter:04d}",
                },
            )

            all_chunks.append(chunk)
            chunk_counter += 1
            search_start = chunk_start + 1

    return all_chunks

In [13]:
chunks = create_all_chunks(grouped_sections, splitter)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3], 1):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("Text:", chunk.page_content[:500])
    print("Metadata:", chunk.metadata)

Total chunks: 1170
CHUNK 1
Text: Terms used in this guideline ................................................................................................................. 48
Recommendations for research .................................................................................................49
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.14 Managing complications', 'page_number': 3, 'page_numbers': [3], 'chunk_id': 'chunk_0001'}
CHUNK 2
Text: 1 Clinical features for distinguishing between type 1 diabetes and other types of diabetes ........ 49
2 The use of C-peptide in diagnosing diabetes ................................................................................. 49
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.14 Managing complications', 'page_number': 3, 'page_numbers': [3], 'chunk_id': 'chunk_0002'}
CHUNK 3
Text: 3 Use of routinely collected real-world data to examine the effectiveness and cost
effectiveness of continuous glu

## 8. Validate chunk metadata

In [14]:
required_fields = [
    "document_name",
    "section",
    "page_number",
    "page_numbers",
    "chunk_id",
]

for i, chunk in enumerate(chunks):
    for field in required_fields:
        assert field in chunk.metadata, f"Missing '{field}' in chunk {i}"

chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk IDs found"

print("Metadata validation passed.")
print("Unique chunks:", len(chunks))

document_counts = Counter(
    chunk.metadata["document_name"] for chunk in chunks
)

print("\nChunks per document:")
for document, count in document_counts.items():
    print("-", document, ":", count)

Metadata validation passed.
Unique chunks: 1170

Chunks per document:
- Type-1 diabetes.pdf : 393
- Type-2 diabetes.pdf : 777


## 9. Generate embeddings and create the Chroma vector store

In [15]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = FastEmbedEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Chroma metadata must use scalar values, so convert page_numbers to a string.
cleaned_chunks = []

for doc in chunks:
    clean_meta = {}

    for key, value in doc.metadata.items():
        if value is None:
            clean_meta[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean_meta[key] = value
        else:
            clean_meta[key] = str(value)

    doc.metadata = clean_meta
    cleaned_chunks.append(doc)

persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vector_db = Chroma.from_documents(
    documents=cleaned_chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_name="diabetes_educational_rag",
)

stored_count = vector_db._collection.count()

print("Input chunks :", len(chunks))
print("Stored vectors:", stored_count)

assert stored_count == len(chunks), "Chroma count does not match input chunks."
print("Chroma indexing completed successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Input chunks : 1170
Stored vectors: 1170
Chroma indexing completed successfully.


## 10. Retrieval test

In [16]:
def _extract_pages(chunk_pages, fallback_page):
    if isinstance(chunk_pages, list):
        return [
            int(p)
            for p in chunk_pages
            if str(p).strip().lstrip("-").isdigit()
        ]

    if isinstance(chunk_pages, str):
        found = re.findall(r"\d+", chunk_pages)
        if found:
            return [int(p) for p in found]

    return [fallback_page] if fallback_page is not None else []


def retrieve_with_similarity(question, k=4):
    return vector_db.similarity_search_with_relevance_scores(question, k=k)


def print_retrieval_results(question, k=4):
    results = retrieve_with_similarity(question, k=k)

    print(f"QUESTION: {question}\n")

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"Rank {rank}")
        print("Document :", doc.metadata.get("document_name"))
        print("Page     :", doc.metadata.get("page_number"))
        print("Pages    :", doc.metadata.get("page_numbers"))
        print("Section  :", doc.metadata.get("section"))
        print("Chunk ID :", doc.metadata.get("chunk_id"))
        print("Score    :", round(score, 4))
        print("Text     :", doc.page_content[:500].replace("\n", " "), "...")
        print()

    return results

In [17]:
results = print_retrieval_results(
    "What are the diagnostic criteria for diabetes?",
    k=4,
)

QUESTION: What are the diagnostic criteria for diabetes?

Rank 1
Document : Type-1 diabetes.pdf
Page     : 6
Pages    : [6]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0015
Score    : 0.6632
Text     : Initial diagnosis 1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes typically (but not always) have 1 or more of: • ketosis • rapid weight loss • age of onset under 50 years ...

Rank 2
Document : Type-1 diabetes.pdf
Page     : 7
Pages    : [7]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0017
Score    : 0.6068
Text     : 2022] 1.1.2 Do not use age or BMI alone to exclude or diagnose type 1 diabetes in adults. [2022] 1.1.3 Take into consideration the possibility of other diabetes subtypes and revisit the diagnosis at subsequent clinical reviews. Carry out further investigations if there ...

Rank 3
Document : Type-1 diabetes.pdf
Page     : 5
Page

**Note:** the test questions are now hardcoded directly in this notebook instead of parsed
from `RAG_Test_Questions.pdf`. Parsing a PDF table with regex is fragile — it silently breaks
(and produces `0/0` results) whenever the PDF's layout changes. Editing questions here is
also easier: just edit the list below.


In [18]:

supported_questions = [
    {"number": 1,  "question": "What should my HbA1c number be if I have type 1 diabetes?", "expected_source": "type-1.pdf", "expected_page": 18, "expected_keywords": ["48", "6.5"]},
    {"number": 2,  "question": "How often should I get my HbA1c checked?", "expected_source": "type-1.pdf", "expected_page": 17, "expected_keywords": ["3 to 6 months"]},
    {"number": 3,  "question": "What are the signs that someone might have type 1 diabetes?", "expected_source": "type-1.pdf", "expected_page": 6, "expected_keywords": ["ketosis", "weight loss"]},
    {"number": 4,  "question": "Can a doctor tell I have type 1 diabetes just from my age or weight?", "expected_source": "type-1.pdf", "expected_page": 7, "expected_keywords": ["BMI"]},
    {"number": 5,  "question": "What kind of insulin plan do people with type 1 diabetes usually start with?", "expected_source": "type-1.pdf", "expected_page": 24, "expected_keywords": ["basal", "bolus"]},
    {"number": 6,  "question": "When do I need to check my blood sugar more than 10 times a day?", "expected_source": "type-1.pdf", "expected_page": 23, "expected_keywords": ["10 times"]},
    {"number": 7,  "question": "What should be done if someone with diabetes passes out from low blood sugar?", "expected_source": "type-1.pdf", "expected_page": 31, "expected_keywords": ["glucagon"]},
    {"number": 8,  "question": "What blood sugar level should I aim for before an operation?", "expected_source": "type-1.pdf", "expected_page": 38, "expected_keywords": ["5 to 8"]},
    {"number": 9,  "question": "How often should someone with type 2 diabetes get their HbA1c checked?", "expected_source": "type-2.pdf", "expected_page": 12, "expected_keywords": ["HbA1c"]},
    {"number": 10, "question": "Do I need to check my blood sugar every day if I have type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 14, "expected_keywords": ["self-monitoring"]},
    {"number": 11, "question": "What's usually the first medicine given for type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 32, "expected_keywords": ["metformin"]},
    {"number": 12, "question": "What treatment is given if someone has type 2 diabetes and a heart problem?", "expected_source": "type-2.pdf", "expected_page": 40, "expected_keywords": ["heart failure"]},
    {"number": 13, "question": "What treatment options are there for someone with type 2 diabetes who is overweight?", "expected_source": "type-2.pdf", "expected_page": 58, "expected_keywords": ["obesity"]},
    {"number": 14, "question": "What should be checked before starting SGLT-2 medicine?", "expected_source": "type-2.pdf", "expected_page": 70, "expected_keywords": ["SGLT-2"]},
    {"number": 15, "question": "Should I take aspirin to protect my heart if I have type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 119, "expected_keywords": ["antiplatelet", "aspirin"]},
]

unsupported_questions = [
    {"number": 1, "question": "How much do diabetes medicines cost in Egypt?", "reason": "No pricing or local market info in either PDF"},
    {"number": 2, "question": "Is there a cure for diabetes now?", "reason": "Not mentioned in either guideline"},
    {"number": 3, "question": "Can people with type 1 diabetes take weight-loss injections like Ozempic?", "reason": "type-1.pdf does not cover this type of medicine at all"},
    {"number": 4, "question": "Will type 1 diabetes affect my chances of having children?", "reason": "type-1.pdf just points to a separate pregnancy guideline, no real answer given"},
    {"number": 5, "question": "Which brand of blood sugar monitor is the best one to buy?", "reason": "The guidelines don't recommend specific brands or products"},
]

print("Supported questions  :", len(supported_questions))
print("Unsupported questions:", len(unsupported_questions))

Supported questions  : 15
Unsupported questions: 5


In [19]:
def check_match(
    results,
    expected_source,
    expected_page,
    expected_keywords=None,
    page_tolerance=3
):
    """
    Checks whether at least one retrieved chunk:
      1. Comes from the expected guideline.
      2. Is within the expected page tolerance.
      3. Contains ALL expected keywords.

    Returns:
        page_match
        keyword_match
        match
    """

    expected_source = (expected_source or "").lower()

    # Identify expected guideline
    if "type-1" in expected_source:
        expected_tag = "type-1"
    elif "type-2" in expected_source:
        expected_tag = "type-2"
    else:
        expected_tag = expected_source

    expected_keywords = [
        kw.lower().strip()
        for kw in (expected_keywords or [])
        if kw and kw.strip()
    ]

    page_match = False
    keyword_match = False
    full_match = False

    for doc, _score in results:

        # -----------------------------
        # Document / source
        # -----------------------------
        doc_name = doc.metadata.get(
            "document_name",
            ""
        ).lower()

        source_match = expected_tag in doc_name

        if not source_match:
            continue

        # -----------------------------
        # Pages
        # -----------------------------
        chunk_pages = _extract_pages(
            doc.metadata.get("page_numbers"),
            doc.metadata.get("page_number")
        )

        current_page_match = any(
            abs(page - expected_page) <= page_tolerance
            for page in chunk_pages
        )

        if not current_page_match:
            continue

        # We found the expected source + page
        page_match = True

        # -----------------------------
        # Keywords
        # -----------------------------
        chunk_text = doc.page_content.lower()

        if expected_keywords:

            current_keyword_match = all(
                keyword in chunk_text
                for keyword in expected_keywords
            )

        else:
            current_keyword_match = True

        if current_keyword_match:
            keyword_match = True
            full_match = True

            # No need to check remaining chunks
            break

    return {
        "page_match": page_match,
        "keyword_match": keyword_match,
        "match": full_match,
    }

===== SUPPORTED QUESTIONS =====

QUESTION: What should my HbA1c number be if I have type 1 diabetes?

Rank 1
Document : Type-1 diabetes.pdf
Page     : 17
Pages    : [17]
Section  : 1.6 Blood glucose management
Chunk ID : chunk_0078
Score    : 0.6731
Text     : HbA1c measurement and targets Measurement 1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015] 1.6.2 Consider measuring HbA1c levels more often in adults with type 1 diabetes if their blood glucose control is suspected to be changing rapidly; for example, if their ...

Rank 2
Document : Type-1 diabetes.pdf
Page     : 23
Pages    : [23]
Section  : 1.6 Blood glucose management
Chunk ID : chunk_0113
Score    : 0.6476
Text     : Blood glucose targets 1.6.22 Advise adults with type 1 diabetes to aim for: • a fasting plasma glucose level of 5 to 7 mmol/litre on waking and • a plasma glucose level of 4 to 7 mmol/litre before meals at other times of the day. [2015] ...

Rank 3
Document : Type-1 diabetes.p

In [20]:
print("===== UNSUPPORTED QUESTIONS =====\n")

for item in unsupported_questions:
    results = print_retrieval_results(item["question"], k=4)
    top_score = results[0][1] if results else 0

    print("Why unsupported:", item["reason"])
    print("Top-1 similarity:", round(top_score, 4))
    print("=" * 80)

===== UNSUPPORTED QUESTIONS =====

QUESTION: How much do diabetes medicines cost in Egypt?

Rank 1
Document : Type-2 diabetes.pdf
Page     : 54
Pages    : [54]
Section  : 1.16 People with early onset type 2 diabetes
Chunk ID : chunk_0723
Score    : 0.4986
Text     : effective, while adding liraglutide to an SGLT-2 inhibitor and metformin reported an incremental cost-effectiveness ratio (ICER) approaching £20,000 per quality-adjusted life year (QALY) gained. Tirzepatide was not analysed for this population. ...

Rank 2
Document : Type-2 diabetes.pdf
Page     : 125
Pages    : [125]
Section  : 1.45 Antiplatelet therapy
Chunk ID : chunk_1131
Score    : 0.4986
Text     : effective, while adding liraglutide to an SGLT-2 inhibitor and metformin reported an incremental cost-effectiveness ratio (ICER) approaching £20,000 per quality-adjusted life year (QALY) gained. Tirzepatide was not analysed for this population. ...

Rank 3
Document : Type-2 diabetes.pdf
Page     : 41
Pages    : [41]
Section

## Final status

In [21]:
print("===== RAG DAY 1 STATUS =====")
print("PDFs loaded       :", len(PDF_PATHS))
print("Pages loaded      :", len(all_pages))
print("Grouped sections  :", len(grouped_sections))
print("Chunks created    :", len(chunks))
print("Vectors stored    :", vector_db._collection.count())
print("Supported tests   :", len(supported_questions))
print("Unsupported tests :", len(unsupported_questions))

===== RAG DAY 1 STATUS =====
PDFs loaded       : 2
Pages loaded      : 195
Grouped sections  : 118
Chunks created    : 1170
Vectors stored    : 1170
Supported tests   : 15
Unsupported tests : 5


## 12. Generation step (RAG answer, not just retrieval)

Everything above only *retrieves* chunks. This section adds the missing piece: send the retrieved chunks to an LLM and have it answer **only from that text**, or say clearly that the answer is not in the provided documents. This is the real hallucination/abstention test — a low similarity score alone doesn't prove the final answer will be safe.

In [22]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [23]:
import os
from groq import Groq
from getpass import getpass

# Free API key, no credit card needed: sign up at https://console.groq.com
# then Settings -> API Keys -> Create API Key
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

Enter your Groq API key: ··········


In [24]:
RAG_SYSTEM_PROMPT = """You are a medical-guideline assistant. You answer ONLY using the
CONTEXT chunks provided below, which come from NICE diabetes guidelines (NG17 for type 1,
NG28 for type 2).

Rules:
1. Base your answer strictly on the CONTEXT. Do not use outside knowledge, even if you
   know the answer from general medical knowledge.
2. If the CONTEXT does not contain enough information to answer the question, reply
   exactly: "I don't have enough information in the provided documents to answer that."
   Do not guess, estimate, or fill gaps with general knowledge.
3. When you do answer, cite the source document and page number in parentheses, e.g.
   (type-1.pdf, p.18).
4. Keep the answer short and in plain language, as if speaking to a patient.
5. Never invent a page number, statistic, or recommendation that is not literally present
   in the CONTEXT.
"""


def build_context(results):
    """Formats retrieved (doc, score) pairs into a labeled context block for the LLM."""
    blocks = []
    for doc, _score in results:
        source = doc.metadata.get("document_name", "unknown")
        page = doc.metadata.get("page_number", "?")
        blocks.append(f"[Source: {source}, page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


def generate_answer(question, results, model="llama-3.3-70b-versatile"):
    context = build_context(results)
    user_message = f"CONTEXT:\n{context}\n\nQUESTION: {question}"

    response = client.chat.completions.create(
        model=model,
        max_tokens=300,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content

### Quick test: one supported question, one unsupported question

In [25]:
test_q = "What should my HbA1c number be if I have type 1 diabetes?"
results = retrieve_with_similarity(test_q, k=4)
answer = generate_answer(test_q, results)

print("QUESTION:", test_q)
print("\nANSWER:\n", answer)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
test_q_unsupported = "How much do diabetes medicines cost in Egypt?"
results = retrieve_with_similarity(test_q_unsupported, k=4)
answer = generate_answer(test_q_unsupported, results)

print("QUESTION:", test_q_unsupported)
print("\nANSWER:\n", answer)

### Full run: generation over all 15 supported + 5 unsupported questions

For supported questions, check the answer actually states the fact (not just avoids the question). For unsupported questions, the answer should closely match the abstention sentence in the system prompt — any specific number, price, or recommendation appearing here is a hallucination.

In [ ]:
ABSTAIN_PHRASE = "i don't have enough information"

print("===== GENERATION: SUPPORTED QUESTIONS =====\n")
for item in supported_questions:
    results = retrieve_with_similarity(item["question"], k=4)
    answer = generate_answer(item["question"], results)
    abstained = ABSTAIN_PHRASE in answer.lower()
    print(f"Q{item['number']}: {item['question']}")
    print("A:", answer)
    print("Abstained:", abstained, "<-- should be False for supported questions")
    print("=" * 80)

In [ ]:
print("===== GENERATION: UNSUPPORTED QUESTIONS =====\n")
for item in unsupported_questions:
    results = retrieve_with_similarity(item["question"], k=4)
    answer = generate_answer(item["question"], results)
    abstained = ABSTAIN_PHRASE in answer.lower()
    print(f"Q{item['number']}: {item['question']}")
    print("Why unsupported:", item["reason"])
    print("A:", answer)
    print("Abstained:", abstained, "<-- should be True for unsupported questions")
    print("=" * 80)

## 13. Diagnostic: was the official recommendation page actually retrieved?

The generation step can produce a *factually correct* answer while citing a page that isn't the guideline's official recommendation (e.g. a general "Context" section instead of the numbered recommendation). This cell checks, for each supported question, whether `expected_page` was among the retrieved chunk pages at k=4 (what generation used) and at k=6, so you can tell whether the fix is "retrieve more chunks" or something else.

In [ ]:
def diagnose_page_retrieval(item, k_values=(4, 6)):
    """For one question, show whether expected_page appears in the retrieved
    chunk pages at each k, and which chunk/page was actually cited by generation."""
    expected_tag = "type-1" if "type-1" in item["expected_source"] else "type-2"
    expected_page = item["expected_page"]

    print(f"Q{item['number']}: {item['question']}")
    print(f"Expected: {item['expected_source']} p.{expected_page}")

    for k in k_values:
        results = retrieve_with_similarity(item["question"], k=k)
        found_pages = []
        expected_page_present = False

        for doc, score in results:
            doc_name = doc.metadata.get("document_name", "").lower()
            if expected_tag not in doc_name:
                continue
            chunk_pages = _extract_pages(
                doc.metadata.get("page_numbers"),
                doc.metadata.get("page_number"),
            )
            found_pages.extend(chunk_pages)
            if expected_page in chunk_pages:
                expected_page_present = True

        status = "FOUND in top-k" if expected_page_present else "MISSING from top-k"
        print(f"  k={k}: pages retrieved = {sorted(set(found_pages))}  ->  {status}")

    print("=" * 80)


print("===== PAGE-RETRIEVAL DIAGNOSTIC (k=4 vs k=6) =====\n")
for item in supported_questions:
    diagnose_page_retrieval(item)